# 03 - Attention (AI Infra 视角)

本节从 **工程实现** 角度理解 Attention：
- FlashAttention 原理
- GQA/MQA 与 KV Cache 显存
- PyTorch SDPA
- 推理优化

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Attention 核心 (30秒版)

```
Q (Query): 我在找什么
K (Key):   我有什么
V (Value): 实际内容

Attention(Q, K, V) = softmax(Q @ K.T / √d) @ V
```

**计算量**: O(n² × d)，n=序列长度，d=维度

## 2. 标准 Attention 的问题

```
Q @ K.T 产生 (n × n) 矩阵

seq_len=8K, 头数=32:
  注意力矩阵: 8K × 8K × 32 = 2GB (FP32)
  
seq_len=32K:
  注意力矩阵: 32K × 32K × 32 = 32GB!
```

**问题**: 显存和带宽瓶颈

In [2]:
def attention_memory(seq_len, n_heads, dtype_bytes=4):
    """计算注意力矩阵的显存"""
    mem = seq_len * seq_len * n_heads * dtype_bytes
    return mem / 1e9  # GB

print("注意力矩阵显存:")
for seq_len in [2048, 4096, 8192, 16384, 32768]:
    mem = attention_memory(seq_len, n_heads=32)
    print(f"  seq_len={seq_len:5d}: {mem:.1f} GB")

注意力矩阵显存:
  seq_len= 2048: 0.5 GB
  seq_len= 4096: 2.1 GB
  seq_len= 8192: 8.6 GB
  seq_len=16384: 34.4 GB
  seq_len=32768: 137.4 GB


## 3. FlashAttention (重要!)

**核心思想**: 分块计算，减少 HBM 访问

```
标准 Attention:                  FlashAttention:
                                 
1. Q @ K.T → 存入 HBM (n×n)      1. 分块加载 Q, K, V 到 SRAM
2. 从 HBM 读出，softmax           2. 在 SRAM 中计算分块 attention
3. 存入 HBM                       3. 累加结果，输出
4. 读出，@ V                     
5. 存入 HBM                       不存储 n×n 矩阵!

HBM 访问: O(n²)                   HBM 访问: O(n)
```

### 关键技术
1. **Tiling**: 分块到 SRAM
2. **Online Softmax**: 边算边更新 softmax
3. **Recomputation**: 反向传播时重算而不是存储

In [3]:
# PyTorch 内置 FlashAttention (通过 SDPA)
# 不需要手动实现!

q = torch.randn(2, 8, 1024, 64)  # (batch, heads, seq, dim)
k = torch.randn(2, 8, 1024, 64)
v = torch.randn(2, 8, 1024, 64)

# 自动选择最优实现 (FlashAttention if available)
out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print(f"输出: {out.shape}")

# 查看使用的后端
print(f"\n可用后端:")
print(f"  Flash: {torch.backends.cuda.flash_sdp_enabled()}")
print(f"  Memory Efficient: {torch.backends.cuda.mem_efficient_sdp_enabled()}")
print(f"  Math: {torch.backends.cuda.math_sdp_enabled()}")

输出: torch.Size([2, 8, 1024, 64])

可用后端:
  Flash: True
  Memory Efficient: True
  Math: True


### FlashAttention 的限制

- 需要特定 GPU (Ampere+)
- head_dim 必须是 8 的倍数
- 某些 mask 模式不支持
- 不支持 attention dropout (在 FA2 中已解决)

## 4. GQA / MQA (超级重要!)

**目的**: 减少 KV Cache 显存

```
MHA (Multi-Head Attention):
  Q: 32 头
  K: 32 头  ← 每个 Q 头对应一个 KV 头
  V: 32 头
  KV Cache: 32 × 2 = 64 份

GQA (Grouped-Query Attention, 分 4 组):
  Q: 32 头
  K: 8 头   ← 4 个 Q 头共享 1 个 KV 头
  V: 8 头
  KV Cache: 8 × 2 = 16 份 (节省 75%!)

MQA (Multi-Query Attention, 极端情况):
  Q: 32 头
  K: 1 头   ← 所有 Q 头共享 1 个 KV 头
  V: 1 头
  KV Cache: 1 × 2 = 2 份 (节省 96%!)
```

In [1]:
def kv_cache_memory(batch_size, seq_len, n_layers, n_kv_heads, head_dim, dtype_bytes=2):
    """
    计算 KV Cache 显存
    
    KV Cache = 2 (K和V) × batch × seq × layers × heads × head_dim × dtype
    """
    mem = 2 * batch_size * seq_len * n_layers * n_kv_heads * head_dim * dtype_bytes
    return mem / 1e9  # GB

# LLaMA 7B 配置
batch_size = 32
seq_len = 4096
n_layers = 32
head_dim = 128

print(f"KV Cache 显存 (batch={batch_size}, seq={seq_len}):")
print(f"")

for n_kv_heads, name in [(32, 'MHA'), (8, 'GQA (4:1)'), (1, 'MQA')]:
    mem = kv_cache_memory(batch_size, seq_len, n_layers, n_kv_heads, head_dim)
    print(f"  {name:12s}: {mem:.1f} GB")

KV Cache 显存 (batch=32, seq=4096):

  MHA         : 68.7 GB
  GQA (4:1)   : 17.2 GB
  MQA         : 2.1 GB


In [ ]:
# GQA 实现
class GroupedQueryAttention(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads      # Q 头数
        self.n_kv_heads = n_kv_heads  # KV 头数 (更少)
        self.head_dim = dim // n_heads
        
        # Q 投影: n_heads 个头
        self.c_q = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        # K, V 投影: n_kv_heads 个头 (更少!)
        self.c_k = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.c_v = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.c_proj = nn.Linear(dim, dim, bias=False)
    
    def forward(self, x):
        B, T, D = x.shape
        
        q = self.c_q(x).view(B, T, self.n_heads, self.head_dim)
        k = self.c_k(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.c_v(x).view(B, T, self.n_kv_heads, self.head_dim)
        
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        
        # PyTorch SDPA 支持 GQA!
        enable_gqa = self.n_heads != self.n_kv_heads
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=enable_gqa)
        
        return self.c_proj(out.transpose(1, 2).contiguous().view(B, T, -1))

# 测试
gqa = GroupedQueryAttention(dim=512, n_heads=8, n_kv_heads=2)
x = torch.randn(2, 16, 512)
out = gqa(x)
print(f"输入: {x.shape} -> 输出: {out.shape}")
print(f"Q 头: 8, KV 头: 2 (每 4 个 Q 头共享 1 个 KV 头)")

## 5. KV Cache 详解

**目的**: 推理时避免重复计算

```
无 Cache:                       有 Cache:

Step 1: 计算 [A] 的 K,V        Step 1: 计算 [A] 的 K,V, 存入 cache
Step 2: 计算 [A,B] 的 K,V      Step 2: 只算 [B], 用 cache 中的 A
Step 3: 计算 [A,B,C] 的 K,V    Step 3: 只算 [C], 用 cache 中的 A,B

计算量: O(n²)                   计算量: O(n)
```

In [ ]:
class KVCache:
    """
    简化版 KV Cache
    """
    def __init__(self, batch_size, max_seq_len, n_layers, n_kv_heads, head_dim, device='cpu'):
        self.max_seq_len = max_seq_len
        self.n_layers = n_layers
        
        # 预分配 cache: (n_layers, 2, batch, n_kv_heads, max_seq, head_dim)
        self.cache = torch.zeros(
            n_layers, 2, batch_size, n_kv_heads, max_seq_len, head_dim,
            device=device, dtype=torch.float16
        )
        self.pos = 0
    
    def update(self, layer_idx, k, v):
        """插入新的 K, V 并返回完整序列"""
        T_new = k.size(2)  # 新 token 数
        
        # 插入
        self.cache[layer_idx, 0, :, :, self.pos:self.pos+T_new, :] = k
        self.cache[layer_idx, 1, :, :, self.pos:self.pos+T_new, :] = v
        
        # 更新位置 (只在最后一层)
        if layer_idx == self.n_layers - 1:
            self.pos += T_new
        
        # 返回完整的 K, V
        return (
            self.cache[layer_idx, 0, :, :, :self.pos, :],
            self.cache[layer_idx, 1, :, :, :self.pos, :]
        )

# 测试
cache = KVCache(batch_size=1, max_seq_len=1024, n_layers=2, n_kv_heads=4, head_dim=64)
print(f"Cache 显存: {cache.cache.numel() * 2 / 1e6:.1f} MB")
print(f"初始位置: {cache.pos}")

## 6. QK Norm

**目的**: 稳定注意力分数

```python
# nanochat 在 RoPE 之后做 QK Norm
q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)
q, k = norm(q), norm(k)  # QK Norm
```

**原因**: Q·K 点积 = |Q| × |K| × cos(θ)，归一化后只看角度

## 7. 面试常见问题

### Q1: FlashAttention 为什么快?

**答**: 减少 HBM 访问，不存储 O(n²) 的注意力矩阵
- 分块加载到 SRAM
- Online Softmax 边算边更新
- IO-aware，优化内存带宽

---

### Q2: GQA 和 MQA 的区别?

**答**:
- MHA: 每个 Q 头有自己的 KV 头
- GQA: 多个 Q 头共享一个 KV 头 (例如 4:1)
- MQA: 所有 Q 头共享一个 KV 头 (极端情况)

GQA 是 MHA 和 MQA 的折中，效果和速度都不错。

---

### Q3: KV Cache 占多少显存?

**答**: `2 × batch × seq × layers × kv_heads × head_dim × dtype`

7B 模型，batch=1，seq=4096：
- MHA (32 heads): ~4 GB
- GQA (8 heads): ~1 GB

---

### Q4: Causal Mask 是什么?

**答**: 防止看到未来 token
- 下三角矩阵: mask[i][j] = 1 if j <= i else 0
- softmax 前把 mask=0 的位置设为 -inf
- PyTorch: `is_causal=True`

---

### Q5: 为什么要除以 √d?

**答**: 防止点积值过大
- 点积的方差 ∝ d
- 值太大 → softmax 变成 one-hot → 梯度消失
- 除以 √d 让方差稳定在 1

---

### Q6: Prefill 和 Decode 的区别?

**答**:
- **Prefill**: 处理 prompt，并行计算所有 token，填充 KV Cache
- **Decode**: 逐个生成 token，每次只处理 1 个 token，查询 KV Cache

Prefill 是 compute bound，Decode 是 memory bound

## 8. 总结速查表

| 主题 | 要点 |
|------|------|
| **FlashAttention** | 分块计算，减少 HBM 访问，不存 O(n²) 矩阵 |
| **GQA** | 多个 Q 头共享 KV 头，减少 KV Cache |
| **KV Cache** | 缓存历史 K,V，Decode 只算新 token |
| **PyTorch SDPA** | `F.scaled_dot_product_attention()` 自动选最优 |
| **Causal Mask** | `is_causal=True`，不看未来 |
| **QK Norm** | 归一化 Q, K，稳定注意力分数 |

### nanochat Attention 核心代码

```python
# 投影 (GQA: KV 头数更少)
q = self.c_q(x).view(B, T, n_heads, head_dim)
k = self.c_k(x).view(B, T, n_kv_heads, head_dim)
v = self.c_v(x).view(B, T, n_kv_heads, head_dim)

# RoPE + QK Norm
q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)
q, k = norm(q), norm(k)

# KV Cache
if kv_cache is not None:
    k, v = kv_cache.insert_kv(layer_idx, k, v)

# Attention (支持 GQA + Causal)
y = F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=True)
```